# Typesense + LangChain RAG

This notebook loads local text files, splits them into chunks, stores their embeddings in Typesense, retrieves relevant context, and sends that context to Groq for a grounded answer.

Set these values in the project-root `.env` file before running: `TYPESENSE_HOST`, `TYPESENSE_API_KEY`, `TYPESENSE_PORT` (usually `443` for Typesense Cloud), `TYPESENSE_PROTOCOL` (usually `https`), and `GROQ_API_KEY`. Never put API keys directly in a notebook.

## 1. Imports and safe configuration

Typesense is the vector database: it stores document embeddings and finds semantically similar chunks. LangChain connects the loader, splitter, embedding model, Typesense retriever, and Groq LLM into one RAG pipeline.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Typesense
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter

def find_project_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from the project root or a subfolder.')

PROJECT_ROOT = find_project_root()
TEXT_DIRECTORY = PROJECT_ROOT / 'data' / 'text_files'
load_dotenv(PROJECT_ROOT / '.env')

TYPESENSE_HOST = os.getenv('TYPESENSE_HOST')
TYPESENSE_API_KEY = os.getenv('TYPESENSE_API_KEY')
TYPESENSE_PORT = os.getenv('TYPESENSE_PORT', '443')
TYPESENSE_PROTOCOL = os.getenv('TYPESENSE_PROTOCOL', 'https')
TYPESENSE_COLLECTION = os.getenv('TYPESENSE_COLLECTION', 'rag_text_chunks')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

if not TYPESENSE_HOST or not TYPESENSE_API_KEY:
    raise RuntimeError('Set TYPESENSE_HOST and TYPESENSE_API_KEY in .env.')
if not GROQ_API_KEY:
    raise RuntimeError('Set GROQ_API_KEY in .env.')
if not TEXT_DIRECTORY.is_dir():
    raise FileNotFoundError(f'Text directory not found: {TEXT_DIRECTORY}')

C:\Users\work\AppData\Local\Temp\ipykernel_16356\2350435053.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
d:\ai\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: Set TYPESENSE_HOST and TYPESENSE_API_KEY in .env.

## 2. Load and split the knowledge base

`DirectoryLoader` reads every `.txt` file. Splitting long text improves retrieval because Typesense can return the specific passage that answers a question.

In [ ]:
loader = DirectoryLoader(
    str(TEXT_DIRECTORY),
    glob='**/*.txt',
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
)
documents = loader.load()
if not documents:
    raise ValueError(f'No .txt files found in {TEXT_DIRECTORY}')

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f'Loaded {len(documents)} files and created {len(chunks)} chunks.')

## 3. Create the Typesense vector store

Embeddings convert each chunk into a numeric representation of its meaning. `Typesense.from_documents` creates the collection when needed and uploads the text, metadata, and embeddings. Re-running this cell adds documents again; use a new collection name or clear the collection in Typesense before rebuilding a production index.

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

vector_store = Typesense.from_documents(
    documents=chunks,
    embedding=embeddings,
    typesense_client_params={
        'host': TYPESENSE_HOST,
        'port': TYPESENSE_PORT,
        'protocol': TYPESENSE_PROTOCOL,
        'typesense_api_key': TYPESENSE_API_KEY,
        'connection_timeout_seconds': 10,
        # This integration forwards collection settings through client params.
        'typesense_collection_name': TYPESENSE_COLLECTION,
    },
)
print(f'Indexed {len(chunks)} chunks in Typesense collection: {TYPESENSE_COLLECTION}')

## 4. Retrieve relevant chunks

This is the retrieval part of RAG. The user's question is embedded with the same model, then Typesense returns the closest document chunks.

In [ ]:
question = 'What is machine learning?'
retrieved_docs = vector_store.similarity_search(question, k=3)
if not retrieved_docs:
    raise ValueError('No documents were retrieved from Typesense.')

for number, document in enumerate(retrieved_docs, start=1):
    print(f'--- Result {number}: {document.metadata.get("source", "unknown source")} ---')
    print(document.page_content[:500], '\n')

## 5. Generate a grounded answer with Groq

The prompt contains only retrieved Typesense context. This instructs the LLM to answer from your knowledge base and to acknowledge missing information rather than inventing an answer.

In [ ]:
context = '\n\n'.join(document.page_content for document in retrieved_docs)
prompt = PromptTemplate.from_template(
    'Answer only from the supplied context. If the context is insufficient, say so.\n\n'
    'Context:\n{context}\n\nQuestion: {question}\nAnswer:'
)

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model='openai/gpt-oss-20b',
    temperature=0.1,
    max_tokens=512,
)
response = llm.invoke(prompt.format(context=context, question=question))
print(response.content)